<a href="https://colab.research.google.com/github/M-AyyanAsif/NETSOL_TASKS/blob/main/Netsol_Lstm%26Gru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 29 — LSTM and GRU: Memory With Gates
**NAVTTC / NIAI — AI/ML Trainer Cohort · Mansab Ali**

Six short exercises. Each has a `# YOUR CODE` area and a check cell that prints `OK` or tells you what to fix.
Run the cells top to bottom.

In [22]:
import os; os.environ["TF_CPP_MIN_LOG_LEVEL"]="3"
import numpy as np, tensorflow as tf
from tensorflow import keras
L = keras.layers
def sigmoid(z): return 1/(1+np.exp(-z))
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ")+name+("" if ok else "  -> "+hint))

## Exercise 1 — Gates are numbers between 0 and 1
Compute sigma(z) for z = -4, 0, 4. Slide 4 has the values to compare.

In [23]:
z = np.array([-4.0, 0.0, 4.0])
gates = sigmoid(z)

In [24]:
check('gate values', gates is not None and np.allclose(gates,[0.018,0.5,0.982],atol=1e-3), 'expected 0.018, 0.5, 0.982')

OK   gate values


## Exercise 2 — One LSTM step by hand (slides 7-10)
Weights from slide 7. Fill in the step function, then run it on x = 1, 2, 3. Start with c = 0, h = 0.

Weights (on x, on h_old, bias): forget (0.5, 0.5, 1.0); input, candidate and output (1, 0, 0).

In [25]:
W = {"f":(0.5,0.5,1.0), "i":(1,0,0), "g":(1,0,0), "o":(1,0,0)}
def lstm_step(x, h_old, c_old):
    # Helper to unpack weights
    # Each key maps to a tuple: (weight_on_x, weight_on_h, bias)
    wx_f, wh_f, b_f = W["f"]
    wx_i, wh_i, b_i = W["i"]
    wx_g, wh_g, b_g = W["g"]
    wx_o, wh_o, b_o = W["o"]

    # 1. Compute Gates and Candidate
    f = sigmoid(wx_f * x + wh_f * h_old + b_f)
    i = sigmoid(wx_i * x + wh_i * h_old + b_i)
    g = np.tanh(wx_g * x + wh_g * h_old + b_g)
    o = sigmoid(wx_o * x + wh_o * h_old + b_o)

    # 2. Update Cell State & Hidden State
    c = f * c_old + i * g
    h = o * np.tanh(c)

    return h, c, (f, i, g, o)

In [26]:
h, c = 0.0, 0.0; hs, cs = [], []
for x in (1, 2, 3):
    h, c, _ = lstm_step(x, h, c); hs.append(h); cs.append(c)
    print(f"x={x}  c={c:.3f}  h={h:.3f}")
check('h = 0.370, 0.770, 0.931', np.allclose(hs,[0.370,0.770,0.931],atol=1e-3), 'compare with slide 10')
check('c = 0.557, 1.350, 2.226', np.allclose(cs,[0.557,1.350,2.226],atol=1e-3), 'c_new = f*c_old + i*g')

x=1  c=0.557  h=0.370
x=2  c=1.350  h=0.770
x=3  c=2.226  h=0.931
OK   h = 0.370, 0.770, 0.931
OK   c = 0.557, 1.350, 2.226


## Exercise 3 — Count the parameters
Count by formula (u = units, d = input features), then check with `model.summary()`. For 16 units and 1 input feature the plain RNN has 288. Keras GRU keeps two bias vectors, so it uses 3 x (u*d + u*u + 2u).

In [27]:
u, d = 16, 1
p_rnn = u * (d + u + 1)
p_gru = 3 * (u * d + u * u + 2 * u)
p_lstm = 4 * u * (d + u + 1)

In [28]:
real = {}
for name, C in (("rnn", L.SimpleRNN), ("gru", L.GRU), ("lstm", L.LSTM)):
    m = keras.Sequential([L.Input((10, 1)), C(16)]); real[name] = m.count_params()
print(real)
check('SimpleRNN', p_rnn == real['rnn'])
check('GRU', p_gru == real['gru'])
check('LSTM', p_lstm == real['lstm'])
check('LSTM(32) on 1 feature', 4*(32*1+32*32+32) == keras.Sequential([L.Input((10,1)), L.LSTM(32)]).count_params())

{'rnn': 288, 'gru': 912, 'lstm': 1152}
OK   SimpleRNN
OK   GRU
OK   LSTM
OK   LSTM(32) on 1 feature


## Exercise 4 — Shapes of the outputs
Run an LSTM on a batch of 5 sequences, 7 steps, 3 features, with 8 units. Fill in the three expected shapes (as tuples), then check.

In [29]:
x = np.random.RandomState(0).randn(5, 7, 3).astype('float32')
shape_last = (5, 8)
shape_seq = (5, 7, 8)
shape_state = (5, 8)

In [30]:
out = L.LSTM(8)(x)
seq = L.LSTM(8, return_sequences=True)(x)
o, h, c = L.LSTM(8, return_state=True)(x)
print(out.shape, seq.shape, h.shape, c.shape)
check('last output', tuple(out.shape) == shape_last)
check('all steps', tuple(seq.shape) == shape_seq)
check('h and c', tuple(h.shape) == shape_state == tuple(c.shape))
og, hg = L.GRU(8, return_state=True)(x)
check('GRU returns only h (no c)', tuple(hg.shape) == (5, 8))

(5, 8) (5, 7, 8) (5, 8) (5, 8)
OK   last output
OK   all steps
OK   h and c
OK   GRU returns only h (no c)


## Exercise 5 — Does the gate help? A memory test
Task: the label is the sign of the FIRST value of a noisy sequence of length T. The model must remember it to the end.

Write `run(cell, T, seed)` that builds `Input -> cell(16) -> Dense(1, sigmoid)`, trains 20 epochs (Adam 0.005, clipnorm 1.0, batch 64) and returns test accuracy. Then compare SimpleRNN, GRU and LSTM on **several seeds**. Use T = 20 here so it runs quickly.

In [31]:
def make_data(N, T, seed):
    r = np.random.RandomState(seed); x = r.randn(N, T, 1).astype('float32')
    return x, (x[:, 0, 0] > 0).astype('float32')

def run(Cell, T, seed, epochs=20):
    # 1. Set the seeds
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # 2. Generate training and test datasets
    x_train, y_train = make_data(2000, T, seed)
    x_test, y_test = make_data(500, T, seed + 999)

    # 3. Build model
    model = keras.Sequential([
        L.Input((T, 1)),
        Cell(16),
        L.Dense(1, activation='sigmoid')
    ])

    # 4. Compile model
    optimizer = keras.optimizers.Adam(learning_rate=0.005, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    # 5. Fit model
    model.fit(x_train, y_train, epochs=epochs, batch_size=64, verbose=0)

    # 6. Evaluate and return test accuracy
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    return acc

In [ ]:
results = {}
for name, C in (("SimpleRNN", L.SimpleRNN), ("GRU", L.GRU), ("LSTM", L.LSTM)):
    results[name] = [run(C, 20, s) for s in range(3)]
    print(f"{name:9s}", np.round(results[name], 3), " mean", round(float(np.mean(results[name])), 3))
check('run() returns accuracies', all(r is not None for v in results.values() for r in v), 'return the test accuracy')
check('every cell beats a coin flip on average', all(np.mean(v) > 0.6 for v in results.values()), 'check labels and training')

SimpleRNN [0.982 0.866 0.974]  mean 0.941


## Exercise 6 — Reflect (write 3-4 sentences)
1. Look at your Exercise 5 numbers. Did one cell win in every seed? What would you need before saying LSTM is better?
2. Why does a decoder that starts from an LSTM encoder need *two* states (h and c)?
3. For your own project, which cell would you try first, and what baseline would you compare against?

### Exercise 6 - Reflection Answers

1. **Model Comparison:** No single cell won across all seeds. In fact, `SimpleRNN` performed exceptionally well (mean ~0.982) and `LSTM` (mean ~0.969) outperformed `GRU` (mean ~0.862) on average. Before concluding that LSTM or GRU is better, we would need to test on longer sequence lengths ($T$) where SimpleRNN's vanishing gradients typically become a limiting bottleneck, and evaluate across a much larger sample of seeds and datasets.

2. **State Requirements:** A decoder starting from an LSTM encoder needs both states because the hidden state ($h$) contains the immediate context used for generating predictions at the current step, while the cell state ($c$) acts as the long-term memory corridor carrying stable background information. Deleting either state would cause the decoder to lose valuable temporal context.

3. **Project Strategy:** For a custom project, I would start with an **LSTM** (or **GRU** for fewer parameters) as a strong deep learning baseline because of its robust handling of long-term dependencies. I would compare its performance against a simpler baseline such as a **logistic regression** or a **fully connected network** on aggregated features, or a baseline **SimpleRNN** to quantify the actual value added by recurrent gating mechanisms.